# Divergence and curl: reading Maxwell's equations

**Professor Nano Labs · Electromagnetism**

**Try it live in your browser:** [The divergence and curl lab](https://tonylowgroup.github.io/Professor-Nano-Labs/labs/divergence-curl/)
**Watch first:** [Why are Maxwell equations written in terms of curl and divergence?](https://www.youtube.com/watch?v=K8JEA-S46bM)
**Watch later:** [What makes a vector field?](https://www.youtube.com/watch?v=S7ShN3GIu9Q) · [Derive Coulomb's law from Maxwell equations](https://www.youtube.com/watch?v=lXF7HDVrCRI) · [Derive Biot Savart's law from Maxwell equations](https://www.youtube.com/watch?v=bXslyzSAhv4) · [The curl of gradient is zero!](https://www.youtube.com/watch?v=PC1ZUn1qqNU) · [Divergence of curl is zero!](https://www.youtube.com/watch?v=0HWpuuzlpVQ)

Maxwell's equations ask two questions of each field: how much springs out of a point (the divergence) and how much swirls round it (the curl),

$$\nabla\cdot\mathbf E=\frac{\rho}{\varepsilon_0},\qquad \nabla\cdot\mathbf B=0,\qquad \nabla\times\mathbf E=-\frac{\partial\mathbf B}{\partial t},\qquad \nabla\times\mathbf B=\mu_0\mathbf J+\mu_0\varepsilon_0\frac{\partial\mathbf E}{\partial t}.$$

This notebook

1. measures the flux out of a small box and the circulation round it, and checks that, divided by the area, they give the divergence and curl;
2. integrates the field of point charges over spheres (Gauss's law) and the field of wires round loops (Ampère's law), off-centre too;
3. checks numerically that the curl of a gradient and the divergence of a curl are zero.

It uses the same fields and settings as the lab, and checks every number quoted in the lab's "Predict first" answers. It needs NumPy, SciPy and Matplotlib. **Run all cells from the top.**

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from scipy.integrate import quad, dblquad

g = lambda x, y: np.exp(-(x**2 + y**2) / 2)
FIELDS = {
    'source': lambda x, y: (x, y),
    'vortex': lambda x, y: (-y, x),
    'shear':  lambda x, y: (y, 0 * x),
    'saddle': lambda x, y: (x, -y),
    'river':  lambda x, y: (1 - y**2 / 4, 0 * x),
    'fading': lambda x, y: (1.5 * x * g(x, y), 1.5 * y * g(x, y)),
    'charge': lambda x, y: (x / (x**2 + y**2), y / (x**2 + y**2)),       # a line charge, seen end-on
    'drain':  lambda x, y: (-y / (x**2 + y**2), x / (x**2 + y**2)),      # a whirlpool
}

fig, axs = plt.subplots(2, 4, figsize=(12, 6))
X, Y = np.meshgrid(np.linspace(-1.9, 1.9, 15), np.linspace(-1.9, 1.9, 15))
for ax, (name, F) in zip(axs.flat, FIELDS.items()):
    with np.errstate(invalid='ignore', divide='ignore'): U, V = F(X + 1e-9, Y + 1e-9)
    M = np.hypot(U, V) + 1e-12
    ax.quiver(X, Y, U / M * np.tanh(M), V / M * np.tanh(M), color='0.35'); ax.set_title(name); ax.set_aspect('equal'); ax.set_xticks([]); ax.set_yticks([])
plt.tight_layout(); plt.show()

## 1 · The flux box and the paddle wheel

The flux out of a square box, $\oint\mathbf F\cdot\hat{\mathbf n}\,ds$, and the circulation round it, $\oint\mathbf F\cdot d\mathbf l$, are done side by side with SciPy's adaptive `quad`. Here the divergence and curl come from finite differences, independently of the formulas in the lab.

In [ ]:
def box(F, cx, cy, side):
    h = side / 2
    c = [(cx - h, cy - h), (cx + h, cy - h), (cx + h, cy + h), (cx - h, cy + h)]
    sides = []
    for i in range(4):
        (x1, y1), (x2, y2) = c[i], c[(i + 1) % 4]
        dx, dy = x2 - x1, y2 - y1
        P = lambda s: (x1 + s * dx, y1 + s * dy)
        flux = quad(lambda s: F(*P(s))[0] * dy - F(*P(s))[1] * dx, 0, 1, limit=200)[0]   # normal (dy, -dx): outward going anticlockwise
        circ = quad(lambda s: F(*P(s))[0] * dx + F(*P(s))[1] * dy, 0, 1, limit=200)[0]
        sides.append((flux, circ))
    flux, circ = sum(s[0] for s in sides), sum(s[1] for s in sides)
    return dict(sides=sides, flux=flux, circ=circ, div=flux / side**2, curl=circ / side**2)

def div_curl(F, x, y, e=1e-5):
    dFx_dx = (F(x + e, y)[0] - F(x - e, y)[0]) / (2 * e); dFy_dy = (F(x, y + e)[1] - F(x, y - e)[1]) / (2 * e)
    dFy_dx = (F(x + e, y)[1] - F(x - e, y)[1]) / (2 * e); dFx_dy = (F(x, y + e)[0] - F(x, y - e)[0]) / (2 * e)
    return dFx_dx + dFy_dy, dFy_dx - dFx_dy

print(f'{"field":8s} {"flux":>8s} {"circ":>8s} {"flux/A":>8s} {"circ/A":>8s}   finite differences at the centre')
for name, F in FIELDS.items():
    b = box(F, 0.8, 0.5, 0.6); d, c = div_curl(F, 0.8, 0.5)
    print(f'{name:8s} {b["flux"]:8.4f} {b["circ"]:8.4f} {b["div"]:8.4f} {b["curl"]:8.4f}   div {d:7.4f}, curl {c:7.4f}')

# for a smooth field that is not linear, flux/area tends to the divergence as the box shrinks
for side in [1.0, 0.4, 0.1, 0.02]:
    print(f'fading source at (1, 0.6), box {side}: flux/area = {box(FIELDS["fading"], 1, 0.6, side)["div"]:.6f}')
print('divergence there:', round(div_curl(FIELDS['fading'], 1, 0.6)[0], 6))

### Questions 1 to 4

1. The shear flow $(y,0)$: every arrow points right, yet the curl is $-1$, so the wheel turns clockwise at 0.5 rad per unit time.
2. The whirlpool $(-y,x)/r^2$: zero curl away from the drain; circulation $2\pi$ round any box that encloses it.
3. The fading source: a sink beyond $r=\sqrt2$, with divergence $-0.23$ at $x=1.6$.
4. The saddle: 0.18 in through top and bottom and out through each side, for a box 0.6 across at the centre.

In [ ]:
b = box(FIELDS['shear'], 0.8, 0.5, 0.6)
print(f'Q1 shear: curl {b["curl"]:.4f}, circulation {b["circ"]:.4f}, wheel rate {b["curl"] / 2:.2f}')
assert abs(b['curl'] + 1) < 1e-9 and abs(b['circ'] + 0.36) < 1e-9

off, on = box(FIELDS['drain'], 1, 0.5, 0.6), box(FIELDS['drain'], 0.1, 0.1, 0.6)
print(f'Q2 whirlpool: circulation off the drain {off["circ"]:.2e}, round it {on["circ"]:.4f} (2 pi = {2 * np.pi:.4f})')
assert abs(off['circ']) < 1e-8 and abs(on['circ'] - 2 * np.pi) < 1e-6
assert all(abs(box(FIELDS['drain'], 0.05, -0.1, s)['circ'] - 2 * np.pi) < 1e-6 for s in [0.3, 1.0, 2.0])

dv = div_curl(FIELDS['fading'], 1.6, 0)[0]; b = box(FIELDS['fading'], 1.6, 0, 0.6)
print(f'Q3 fading source at x = 1.6: divergence {dv:.4f}, flux out of the box {b["flux"]:.4f}')
assert round(dv, 2) == -0.23 and b['flux'] < 0
r0 = np.sqrt(2); print('   divergence changes sign at r =', round(r0, 2), '| just inside:', round(div_curl(FIELDS['fading'], r0 - 0.01, 0)[0], 4), ' just outside:', round(div_curl(FIELDS['fading'], r0 + 0.01, 0)[0], 4))

b = box(FIELDS['saddle'], 0, 0, 0.6)
print('Q4 saddle, flux through bottom, right, top, left:', [round(s[0], 3) for s in b['sides']], ' net', round(b['flux'], 12), ' circulation', round(b['circ'], 12))
assert [round(s[0], 2) for s in b['sides']] == [-0.18, 0.18, -0.18, 0.18] and abs(b['flux']) < 1e-12 and abs(b['circ']) < 1e-12

## 2 · Gauss's law and Ampère's law

Point charges in three dimensions, in units where $q/\varepsilon_0=1$: $\mathbf E=q\,\hat{\mathbf r}/4\pi r^2$. The flux through a sphere is done here as a full two-dimensional surface integral with `dblquad`, not with the symmetry trick the lab uses. Wires along $z$, in units where $\mu_0I=1$: $\mathbf B=\hat{\boldsymbol\varphi}/2\pi r$, integrated round circles with `quad`.

In [ ]:
def E(charges, p):
    out = np.zeros(3)
    for (x, y, q) in charges:
        d = p - np.array([x, y, 0.0]); out += q * d / (4 * np.pi * np.linalg.norm(d)**3)
    return out

def sphere_flux(charges, c, R):
    c = np.array([c[0], c[1], 0.0])
    def f(phi, th):
        n = np.array([np.sin(th) * np.cos(phi), np.sin(th) * np.sin(phi), np.cos(th)])
        return E(charges, c + R * n) @ n * R * R * np.sin(th)
    return dblquad(f, 0, np.pi, 0, 2 * np.pi, epsabs=1e-9, epsrel=1e-9)[0]

def B(wires, x, y):
    bx = by = 0.0
    for (wx, wy, I) in wires:
        dx, dy = x - wx, y - wy; r2 = dx * dx + dy * dy; bx -= I * dy / (2 * np.pi * r2); by += I * dx / (2 * np.pi * r2)
    return bx, by

def loop_circ(wires, c, R):
    f = lambda a: np.dot(B(wires, c[0] + R * np.cos(a), c[1] + R * np.sin(a)), (-R * np.sin(a), R * np.cos(a)))
    return quad(f, 0, 2 * np.pi, limit=400, epsabs=1e-11)[0]

ONE, PAIR = [(0, 0, 1)], [(-0.6, 0, 1), (0.6, 0, -1)]
WIRE = [(0, 0, 1)]
for c, R in [((0, 0), 0.5), ((0, 0), 1.0), ((0.3, 0.2), 0.8), ((1.2, 0), 0.4)]:
    print(f'sphere at {c}, radius {R}: flux of one charge {sphere_flux(ONE, c, R):.6f}, of the pair {sphere_flux(PAIR, c, R):.6f};  loop round a wire {loop_circ(WIRE, c, R):.6f}')

### Questions 5 to 7

5. Doubling the sphere round a charge from radius 0.5 to 1: the flux stays $q/\varepsilon_0$; the field on it falls from 0.318 to 0.080.
6. A sphere of radius 1 round the $\pm$ pair: zero flux, though the outward field reaches 0.47.
7. A loop of radius 0.5 centred at $(1, 0.6)$, beside the wire: zero circulation.

In [ ]:
for R in [0.5, 1.0]:
    fl = sphere_flux(ONE, (0, 0), R); En = E(ONE, np.array([R, 0, 0]))[0]
    print(f'Q5 radius {R}: flux {fl:.6f}, field on the sphere {En:.4f}, field x area {En * 4 * np.pi * R * R:.6f}')
assert abs(sphere_flux(ONE, (0, 0), 0.5) - 1) < 1e-7 and abs(sphere_flux(ONE, (0, 0), 1) - 1) < 1e-7
assert round(E(ONE, np.array([0.5, 0, 0]))[0], 3) == 0.318 and round(E(ONE, np.array([1, 0, 0]))[0], 3) == 0.080

th = np.linspace(0, 2 * np.pi, 3601)
rim = np.array([E(PAIR, np.array([np.cos(t), np.sin(t), 0])) @ np.array([np.cos(t), np.sin(t), 0]) for t in th])
print(f'Q6 pair: flux {sphere_flux(PAIR, (0, 0), 1):.2e}; outward field round the rim from {rim.min():.4f} to {rim.max():.4f}')
assert abs(sphere_flux(PAIR, (0, 0), 1)) < 1e-7 and round(rim.max(), 2) == 0.47 and round(rim.min(), 2) == -0.47
print('   slide the sphere to hold only the + charge:', round(sphere_flux(PAIR, (-0.6, 0), 0.3), 6))

ci = loop_circ(WIRE, (1.0, 0.6), 0.5)
Bt = [np.dot(B(WIRE, 1 + 0.5 * np.cos(a), 0.6 + 0.5 * np.sin(a)), (-np.sin(a), np.cos(a))) for a in th]
print(f'Q7 loop beside the wire: circulation {ci:.2e}; field along the loop from {min(Bt):.3f} to {max(Bt):.3f}')
assert abs(ci) < 1e-9 and min(Bt) < -0.2 and max(Bt) > 0.05
assert all(abs(loop_circ(WIRE, c, R) - 1) < 1e-9 for c, R in [((0.3, 0.2), 0.5), ((0, 0), 1.5), ((-0.2, 0.1), 0.3)])

# the field of a wire and of a charge from the two laws, against the radius
r = np.linspace(0.2, 2, 50)
plt.figure(figsize=(6.5, 3.2))
plt.loglog(r, [E(ONE, np.array([x, 0, 0]))[0] for x in r], label='E of a charge: 1/4πr²')
plt.loglog(r, [B(WIRE, x, 0)[1] for x in r], label='B of a wire: 1/2πr')
plt.xlabel('r'); plt.legend(); plt.tight_layout(); plt.show()

## 3 · Two zeros

A landscape $\varphi$: a hill of height 1.2 at $(-0.8, 0.5)$ and a valley of depth 1 at $(0.9, -0.45)$, both Gaussians of width 0.6. The downhill field $-\nabla\varphi$ and the contour field $\nabla\times(\varphi\hat{\mathbf z})=(\partial_y\varphi,-\partial_x\varphi)$ are made here by finite differences of $\varphi$ itself.

In [ ]:
BUMPS, SIG = [(-0.8, 0.5, 1.2), (0.9, -0.45, -1.0)], 0.6
phi = lambda x, y: sum(a * np.exp(-((x - bx)**2 + (y - by)**2) / (2 * SIG**2)) for bx, by, a in BUMPS)
e = 1e-6
dphi = lambda x, y: ((phi(x + e, y) - phi(x - e, y)) / (2 * e), (phi(x, y + e) - phi(x, y - e)) / (2 * e))
GRAD = lambda x, y: tuple(-v for v in dphi(x, y))
CURL = lambda x, y: (dphi(x, y)[1], -dphi(x, y)[0])

rng = np.random.default_rng(0)
worst_g = worst_c = 0
for _ in range(20):
    cx, cy, s = rng.uniform(-1.4, 1.4), rng.uniform(-1.4, 1.4), rng.uniform(0.2, 1.2)
    worst_g = max(worst_g, abs(box(GRAD, cx, cy, s)['circ'])); worst_c = max(worst_c, abs(box(CURL, cx, cy, s)['flux']))
print(f'20 random boxes: largest circulation of the downhill field {worst_g:.1e}; largest flux of the contour field {worst_c:.1e}')
assert worst_g < 1e-7 and worst_c < 1e-7

b = box(GRAD, -0.8, 0.5, 0.6)
print(f'Q8 box on the hilltop: flux out {b["flux"]:.4f}, circulation {b["circ"]:.1e}')
assert round(b['flux'], 1) == 2.1 and abs(b['circ']) < 1e-7

xs = np.linspace(-2, 2, 121); XX, YY = np.meshgrid(xs, xs)
fig, axs = plt.subplots(1, 2, figsize=(10, 4.4))
for ax, (title, F) in zip(axs, [('downhill, −∇φ', GRAD), ('along the contours, ∇×(φẑ)', CURL)]):
    ax.contour(XX, YY, phi(XX, YY), levels=[-0.65, -0.4, -0.15, 0.15, 0.4, 0.65, 0.9], cmap='coolwarm')
    Xq, Yq = np.meshgrid(np.linspace(-1.9, 1.9, 17), np.linspace(-1.9, 1.9, 17)); U, V = F(Xq, Yq)
    ax.quiver(Xq, Yq, U, V, color='0.3'); ax.set_title(title); ax.set_aspect('equal')
plt.tight_layout(); plt.show()

In [ ]:
print('notebook OK: every assertion passed')